# Import Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import decoupler as dc

from Utils import plot_violin_box_combo

import pygenelab as pgl

# Load Data

In [5]:
# load female dataset
female_adata_path = "/ocean/projects/cis240075p/asachan/datasets/TA_muscle/public_datasets/SKM_multimodal_ageing/objects/tmp/myofibers_female.h5ad"
female_adata = sc.read_h5ad(female_adata_path)

# apply filters
female_adata_subset = pgl.subset_adata_by_obs(female_adata, filters={"age": [34, 80]})

# Load Genesets

In [6]:
# get geneset
geneset_path = "metabolism_enriched_pathways_F.csv"
geneset = pd.read_csv(geneset_path)

# apply filters
geneset = geneset[["source", "target"]].copy()
geneset["target"] = geneset["target"].str.upper()

# Score

In [7]:
%%time

# get scores for all the pathways
dc.mt.aucell(
    female_adata_subset,
    geneset,
    raw=False,
    verbose=True
)

2026-06-05 00:53:40 | [INFO] aucell - Running aucell
2026-06-05 00:53:41 | [INFO] Extracted omics mat with 58955 rows (observations) and 48355 columns (features)
2026-06-05 00:53:41 | [WARNING] 6249 features of mat are empty, they will be removed
2026-06-05 00:53:42 | [WARNING] weight not found in net.columns, adding it as:
net['weight'] = 1
2026-06-05 00:53:42 | [INFO] Network has 908 unique features and 8 unique sources
2026-06-05 00:53:42 | [INFO] aucell - calculating 8 AUCs for 42106 targets across 58955 observations, categorizing features at rank=2106
100%|██████████| 58955/58955 [02:53<00:00, 340.40it/s] 
2026-06-05 00:56:36 | [INFO] aucell - done


CPU times: user 2h 55min 50s, sys: 6.19 s, total: 2h 55min 56s
Wall time: 2min 55s


In [8]:
female_adata_subset.obsm["score_aucell"].head()

,HALLMARK_FATTY_ACID_METABOLISM,HALLMARK_XENOBIOTIC_METABOLISM,REACTOME_GLYCOGEN_METABOLISM,REACTOME_GLYCOSAMINOGLYCAN_METABOLISM,REACTOME_METABOLISM_OF_CARBOHYDRATES,REACTOME_PHOSPHOLIPID_METABOLISM,REACTOME_PYRUVATE_METABOLISM,WP_PURINE_METABOLISM
CELL2785_N1_1_1_6_1,0.072756,0.064834,0.143281,0.027699,0.047619,0.067831,0.073262,0.065851
CELL1152_N1_1_1_6_1,0.043863,0.050503,0.129316,0.027591,0.044431,0.066136,0.129939,0.060057
CELL312_N2_1_1_6_1,0.065708,0.047843,0.209370,0.031456,0.045240,0.060427,0.135227,0.062008
CELL2659_N1_1_1_6_1,0.065877,0.036341,0.126746,0.025438,0.032224,0.048645,0.084490,0.076479
CELL303_N2_1_1_6_1,0.055026,0.037972,0.228980,0.013844,0.042532,0.075232,0.049287,0.060265


# Plot & Save

In [9]:
from pathlib import Path

# create output folder
output_dir = Path("Output")
output_dir.mkdir(parents=True, exist_ok=True)

# get a list of pathways
pathways = geneset["source"].unique()

# loop through pathways
for pathway in pathways:
    
    # store in obs
    female_adata_subset.obs[pathway] = female_adata_subset.obsm["score_aucell"][pathway].copy()

    # get group score df
    group_score_df = pgl.prepare_group_score_df(female_adata_subset, group_col="age", value_col=pathway)

    # get cliffs delta
    cliffs_delta_df, _ = pgl.calculate_score_cliffs_delta(adata=female_adata_subset,
                                                          score_col=pathway,
                                                          group_col="age",
                                                          group1=80,
                                                          group2=34)
    cliffs_delta = cliffs_delta_df["cliffs_delta"].iloc[0]
    higher_group = cliffs_delta_df["higher_group"].iloc[0]

    # plot
    PALETTE_34_80 = {
        "34": "#F71480",
        "80": "#FA98C7"}
    
    fig = plot_violin_box_combo(group_score_df, 
                                x_var="age", 
                                y_var=pathway,
                                x_ticks=["34", "80"],
                                palette=PALETTE_34_80,
                                title=f"{pathway} Activity Score",
                                show_scatter=False,
                                show_pvalue=False,
                                group_spacing=0.8,
                                delta_label=f"Cliff's δ = {round(cliffs_delta, 3)} (Higher in Age: {higher_group})")
    
    # save
    svg_path = output_dir / f"Human_F_{pathway}_Score.svg"
    pgl.save_editable_svg(fig, output_path=svg_path)

<Figure size 640x480 with 0 Axes>